In [ ]:
import random
from datasets import load_dataset, Dataset
import pandas as pd
from typing import Iterator, Dict, Any

# ============================================================================
# 💡 데이터셋 정보: Epitech/gemma3-military_drone
# 🤖 데이터셋 의미: 군사 드론 관련 텍스트 데이터셋으로 보입니다.
#    이는 드론 운영, 임무 수행, 기술적 측면에 대한 '질문-답변' 또는
#    '상황-결과' 형태의 텍스트 패턴을 학습시키는 데 사용될 수 있습니다.
# 🎯 실습 목표: 주어진 '입력(input)' 컨텍스트를 바탕으로 모델이 어떤 '출력(output)'
#     을 생성했는지 분석하고, 프롬프트 엔지니어링 관점에서 데이터를 재구성해봅니다.
# ============================================================================

# 상수 정의
DATASET_NAME = "Epitech/gemma3-military_drone"
TRAIN_SPLIT = "train"
SAMPLE_COUNT = 10  # 재미있게 테스트할 샘플 개수 설정! 너무 많으면 오래 걸릴 수 있어요!

# ----------------------------------------------------------------------------
# 🚀 1. 데이터 로드 (스트리밍 최적화 및 오류 처리)
# ----------------------------------------------------------------------------
print("🚀 [단계 1/3] 데이터 로딩을 시작합니다. 스트리밍 방식을 먼저 시도합니다...")

try:
    # 스트리밍 모드 시도: 대용량 데이터셋을 효율적으로 처리하기 위함!
    dataset = load_dataset(DATASET_NAME, split=TRAIN_SPLIT, streaming=True)
    print("✨ 성공! 스트리밍 모드(Streaming)로 데이터셋을 로드했습니다. 메모리 효율 만점!")

except Exception as e:
    # 스트리밍이 불안정하거나 환경 문제로 실패할 경우 폴백 처리
    print(f"⚠️ 스트리밍 로딩 실패 감지 ({e}). 일반 다운로드 방식으로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split=TRAIN_SPLIT, streaming=False)
        print("✅ 성공! 일반 Dataset 방식으로 데이터를 다운로드했습니다.")
    except Exception as fallback_e:
        print(f"🚨 치명적 오류: 데이터를 로드할 수 없습니다. {fallback_e}")
        exit()

# ----------------------------------------------------------------------------
# 🧬 2. 데이터 샘플링 및 Iterator 설정
# ----------------------------------------------------------------------------
print("\n⚙️ [단계 2/3] 데이터 샘플링을 준비합니다...")

# Constraint 9 & 16 적용: .take()를 사용하여 제한된 샘플만 가져와 Iterator로 만듭니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)일 가능성이 높음
    print(f"🔍 스트리밍 모드를 감지: 상위 {SAMPLE_COUNT}개 샘플만 살펴볼게요!")
    # 데이터를 iterator로 변환하여 바로 샘플링 시작
    sampled_dataset_iterator: Iterator[Dict[str, Any]] = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)
    print(f"📚 일반 데이터셋 모드 감지: 상위 {SAMPLE_COUNT}개 샘플을 준비합니다!")
    # 일반 dataset 객체이므로 리스트로 변환하여 처리
    sampled_dataset = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))
    # 반복문을 위해 다시 iterator로 변환
    sampled_dataset_iterator = iter(sampled_dataset)


# ----------------------------------------------------------------------------
# ✨ 3. 창의적 실습: AI 프롬프트 분석 및 시뮬레이션
# ----------------------------------------------------------------------------

def generate_contextual_prompt(sample: Dict[str, Any]) -> str:
    """
    주어진 (입력, 출력) 쌍을 분석하여, 모델이 어떤 종류의 태스크를 수행했는지
    하는 가상의 '프롬프트 분석' 과정을 수행합니다. (초급 난이도)
    """
    input_text = sample.get("input", "N/A").strip()
    output_text = sample.get("output", "N/A").strip()

    # 간단한 키워드 분석을 통한 태스크 유추 시뮬레이션
    if "임무" in input_text or "목표" in input_text:
        task_type = "🎯 임무 목표 분석 (Goal Analysis)"
        analysis = f"이 입력은 {input_text[:30]}...에 대한 임무 계획을 요구합니다. 모델은 이를 바탕으로 {output_text[:30]}...를 예상 답변으로 제공했습니다."
    elif "조건" in input_text or "상황" in input_text:
        task_type = "💡 상황 기반 추론 (Contextual Inference)"
        analysis = f"특정 상황 ({input_text[:30]}...)에서 드론의 적절한 후속 조치({output_text[:30]}...)를 추론하는 문제입니다."
    elif "제한" in input_text or "규정" in input_text:
        task_type = "📜 규정/제한 준수 확인 (Constraint Check)"
        analysis = f"규정적 제약 조건({input_text[:30]}...) 하에서, 모델이 준수해야 할 주요 내용을 확인합니다."
    else:
        task_type = "❓ 일반 텍스트 완성 (General Completion)"
        analysis = f"가장 일반적인 구조로, 주어진 컨텍스트({input_text[:30]}...)를 완성하는 연습입니다."

    return f"\n---\n🤖 태스크 종류: {task_type}\n🔍 분석 내용: {analysis}\n-------------------"


print("\n✨ [단계 3/3] 🔥 데이터 분석 실습을 시작합니다! (상위 {}개 샘플 분석)".format(min(SAMPLE_COUNT, 10)))

sample_count = 0
for sample_data in sampled_dataset_iterator:
    sample_count += 1
    
    # 안전한 데이터 추출
    input_context = sample_data.get("input", "입력 컨텍스트 없음")
    output_response = sample_data.get("output", "출력 응답 없음")

    # ⭐️ 핵심 실습: 프롬프트 분석 함수 호출
    analysis_report = generate_contextual_prompt(sample_data)

    print(f"\n================== [샘플 {sample_count}] ==================")
    print(f"🌐 입력 컨텍스트 (Prompt): {input_context}")
    print(f"✅ 예상 출력 (Answer): {output_response}")
    print(analysis_report)

print("\n=========================================================")
print(f"🎉🎉🎉 축하합니다! 총 {sample_count}개의 샘플 분석을 성공적으로 완료했습니다! 🎉🎉🎉")
print("🚀 이 실습을 통해, AI 모델이 단순히 데이터를 '암기'하는 것이 아니라,")
print("    '상황(Context)'을 보고 '추론(Inference)'하여 '응답(Response)'을 만든다는 원리를 이해했습니다!")
print("👏 다음에는 이 패턴을 이용해 직접 프롬프트를 만들어보는 연습을 해보세요!")
print("=========================================================")